In [0]:
import uuid
from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, LongType
)

CATALOG = spark.sql("SELECT current_catalog()").collect()[0][0]
SCHEMA  = "cyber_threat"

BRONZE_NVD  = f"{CATALOG}.{SCHEMA}.bronze_nvd"
BRONZE_EPSS = f"{CATALOG}.{SCHEMA}.bronze_epss"
BRONZE_KEV  = f"{CATALOG}.{SCHEMA}.bronze_kev"
BRONZE_ASSETS = f"{CATALOG}.{SCHEMA}.bronze_assets"
SILVER_NVD  = f"{CATALOG}.{SCHEMA}.silver_nvd"
SILVER_EPSS = f"{CATALOG}.{SCHEMA}.silver_epss"
SILVER_KEV  = f"{CATALOG}.{SCHEMA}.silver_kev"
LOG_TABLE   = f"{CATALOG}.{SCHEMA}.pipeline_execution_logs"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.quarantine_records"

print("Catalog:", CATALOG, "| Schema:", SCHEMA)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    run_id          STRING,
    layer           STRING,
    load_type       STRING,
    source_param    STRING,
    target_table    STRING,
    start_time      TIMESTAMP,
    end_time        TIMESTAMP,
    status          STRING,
    rows_inserted   BIGINT,
    rows_updated    BIGINT,
    error_message   STRING,
    load_timestamp  TIMESTAMP
) USING DELTA
""")

In [0]:
log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("source_param", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("error_message", StringType(), True),
    StructField("load_timestamp", TimestampType(), True),
])

def new_run_id():
    return str(uuid.uuid4())

def write_log(run_id, layer, load_type, source_param, target_table,
              start_time, status, rows_inserted=0, rows_updated=0, error_message=None):
    end_time = datetime.now()
    row = [(run_id, layer, load_type, str(source_param), target_table,
            start_time, end_time, status,
            int(rows_inserted), int(rows_updated),
            error_message, end_time)]
    spark.createDataFrame(row, schema=log_schema) \
         .write.format("delta").mode("append").saveAsTable(LOG_TABLE)

In [0]:
#Schema definiton:

from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType,
    TimestampType, ArrayType, IntegerType, BooleanType
)

# epss schema (csv)
epss_schema = StructType([
    StructField("cve", StringType(), True),
    StructField("epss", DoubleType(), True),
    StructField("percentile", DoubleType(), True),
])

# cortexa asset inventory schema (csv)
asset_schema = StructType([
    StructField("asset_id", StringType(), True),
    StructField("hostname", StringType(), True),
    StructField("ip_address", StringType(), True),
    StructField("software_vendor", StringType(), True),
    StructField("software_product", StringType(), True),
    StructField("software_version", StringType(), True),
    StructField("asset_group", StringType(), True),
    StructField("environment", StringType(), True),
    StructField("business_criticality", StringType(), True),
    StructField("owner_email", StringType(), True),
    StructField("patch_status", StringType(), True),
    StructField("last_patched_date", DateType(), True),
    StructField("last_scanned_date", DateType(), True),
])

# incremental asset file has 2 extra columns
asset_incremental_schema = StructType(
    asset_schema.fields + [
        StructField("change_type", StringType(), True),
        StructField("last_updated", DateType(), True),
    ]
)

# kev schema (json, wrapper object + array)
kev_vuln_schema = StructType([
    StructField("cveID", StringType(), True),
    StructField("vendorProject", StringType(), True),
    StructField("product", StringType(), True),
    StructField("vulnerabilityName", StringType(), True),
    StructField("dateAdded", StringType(), True),  # cast to date in silver
    StructField("shortDescription", StringType(), True),
    StructField("requiredAction", StringType(), True),
    StructField("dueDate", StringType(), True),  # cast to date in silver
    StructField("knownRansomwareCampaignUse", StringType(), True),
    StructField("forensicTriage", StringType(), True),
    StructField("notes", StringType(), True),
    StructField("cwes", ArrayType(StringType()), True),
])

kev_root_schema = StructType([
    StructField("title", StringType(), True),
    StructField("catalogVersion", StringType(), True),
    StructField("dateReleased", TimestampType(), True),
    StructField("count", IntegerType(), True),
    StructField("vulnerabilities", ArrayType(kev_vuln_schema), True),
])

# nvd schema (json, wrapper object + nested array)
nvd_description_schema = StructType([
    StructField("lang", StringType(), True),
    StructField("value", StringType(), True),
])

nvd_cvss_data_schema = StructType([
    StructField("version", StringType(), True),
    StructField("vectorString", StringType(), True),
    StructField("baseScore", DoubleType(), True),
    StructField("baseSeverity", StringType(), True),
    StructField("attackVector", StringType(), True),
    StructField("attackComplexity", StringType(), True),
    StructField("privilegesRequired", StringType(), True),
    StructField("userInteraction", StringType(), True),
    StructField("scope", StringType(), True),
    StructField("confidentialityImpact", StringType(), True),
    StructField("integrityImpact", StringType(), True),
    StructField("availabilityImpact", StringType(), True),
    StructField("accessVector", StringType(), True),  # v2 only
    StructField("accessComplexity", StringType(), True),  # v2 only
    StructField("authentication", StringType(), True),  # v2 only
])

nvd_cvss_metric_schema = StructType([
    StructField("source", StringType(), True),
    StructField("type", StringType(), True),
    StructField("cvssData", nvd_cvss_data_schema, True),
    StructField("baseSeverity", StringType(), True),
    StructField("exploitabilityScore", DoubleType(), True),
    StructField("impactScore", DoubleType(), True),
])

nvd_metrics_schema = StructType([
    StructField("cvssMetricV2", ArrayType(nvd_cvss_metric_schema), True),
    StructField("cvssMetricV30", ArrayType(nvd_cvss_metric_schema), True),
    StructField("cvssMetricV31", ArrayType(nvd_cvss_metric_schema), True),
    StructField("cvssMetricV40", ArrayType(nvd_cvss_metric_schema), True),
])

nvd_weakness_desc_schema = StructType([
    StructField("lang", StringType(), True),
    StructField("value", StringType(), True),
])

nvd_weakness_schema = StructType([
    StructField("source", StringType(), True),
    StructField("type", StringType(), True),
    StructField("description", ArrayType(nvd_weakness_desc_schema), True),
])

nvd_version_schema = StructType([
    StructField("version", StringType(), True),
    StructField("lessThan", StringType(), True),
    StructField("lessThanOrEqual", StringType(), True),
    StructField("versionType", StringType(), True),
    StructField("status", StringType(), True),
])

nvd_affected_data_schema = StructType([
    StructField("vendor", StringType(), True),
    StructField("product", StringType(), True),
    StructField("defaultStatus", StringType(), True),
    StructField("versions", ArrayType(nvd_version_schema), True),
])

nvd_affected_schema = StructType([
    StructField("source", StringType(), True),
    StructField("affectedData", ArrayType(nvd_affected_data_schema), True),
])

nvd_reference_schema = StructType([
    StructField("url", StringType(), True),
    StructField("source", StringType(), True),
    StructField("tags", ArrayType(StringType()), True),
])

nvd_cve_schema = StructType([
    StructField("id", StringType(), True),
    StructField("sourceIdentifier", StringType(), True),
    StructField("published", TimestampType(), True),
    StructField("lastModified", TimestampType(), True),
    StructField("vulnStatus", StringType(), True),
    StructField("descriptions", ArrayType(nvd_description_schema), True),
    StructField("metrics", nvd_metrics_schema, True),
    StructField("weaknesses", ArrayType(nvd_weakness_schema), True),
    StructField("affected", ArrayType(nvd_affected_schema), True),
    StructField("references", ArrayType(nvd_reference_schema), True),
])

nvd_vulnerability_wrapper_schema = StructType([
    StructField("cve", nvd_cve_schema, True),
])

nvd_root_schema = StructType([
    StructField("resultsPerPage", IntegerType(), True),
    StructField("startIndex", IntegerType(), True),
    StructField("totalResults", IntegerType(), True),
    StructField("format", StringType(), True),
    StructField("version", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("vulnerabilities", ArrayType(nvd_vulnerability_wrapper_schema), True),
])